## 1. Locate source code and the Bank Marketing stream


In [ ]:
from pathlib import Path
import os, shutil, zipfile, json, sys, subprocess, time

INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
CODE = WORK_ROOT / "JCAM" / "Code"
RESULTS = CODE / "results" / "exp10_bank"
BOOTSTRAP_ERROR = None

def extract_matching_zip(target_name, destination):
    matches = []
    for path in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(path) as archive:
                if any(name.replace("\\", "/").endswith(target_name) for name in archive.namelist()):
                    matches.append(path)
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError(f"Expected one archive containing {target_name}, found {matches}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)

try:
    source_roots = sorted({path.parent.parent for path in INPUT_ROOT.rglob("run_application.py")
                           if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if not source_roots:
        extracted_source = WORK_ROOT / "_jcam_source"
        extract_matching_zip("Code/kaggle/run_application.py", extracted_source)
        source_roots = sorted({path.parent.parent for path in extracted_source.rglob("run_application.py")
                               if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if len(source_roots) != 1:
        raise RuntimeError(f"Expected one Code source, found {source_roots}")
    shutil.copytree(source_roots[0], CODE, dirs_exist_ok=True)
    inputs = sorted(INPUT_ROOT.rglob("bank_stream.npz"))
    if not inputs:
        extracted_data = WORK_ROOT / "_jcam_bank_input"
        extract_matching_zip("bank_stream.npz", extracted_data)
        inputs = sorted(extracted_data.rglob("bank_stream.npz"))
    if len(inputs) != 1:
        raise RuntimeError(f"Expected one bank_stream.npz, found {inputs}")
    DATA = inputs[0]
    SOURCE_META = DATA.with_suffix(".metadata.json")
    if not SOURCE_META.is_file():
        raise FileNotFoundError(SOURCE_META)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Code:", CODE, "Data:", DATA)
except Exception as error:
    BOOTSTRAP_ERROR = repr(error)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Bootstrap failed:", BOOTSTRAP_ERROR)


## 2. Validate the chronological NPZ


In [ ]:
import hashlib
import numpy as np

RUN_OK = BOOTSTRAP_ERROR is None
STAGES = []
ROUND_PATH = None

def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)

if RUN_OK:
    try:
        source = json.loads(SOURCE_META.read_text(encoding="utf-8"))
        if hashlib.sha256(DATA.read_bytes()).hexdigest() != source["npz_sha256"]:
            raise ValueError("NPZ SHA-256 mismatch")
        with np.load(DATA, allow_pickle=False) as data:
            x, y, g, r, w = [data[k] for k in ("features", "labels", "groups", "radii", "weights")]
            if x.ndim != 3 or y.shape != x.shape[:2] or g.shape != y.shape:
                raise ValueError("Invalid stream shapes")
            if r.shape != (len(x), 3) or w.shape != r.shape:
                raise ValueError("Invalid objective metadata shapes")
            if not np.all(np.isfinite(x)) or not np.all(np.isin(y, [-1, 1])):
                raise ValueError("Invalid features or labels")
            if not np.all((g == 0).any(axis=1) & (g == 1).any(axis=1)):
                raise ValueError("Both age groups are required in every batch")
            if not np.allclose(w.sum(axis=1), 1) or np.any(w < 0) or np.any(r < 0):
                raise ValueError("Invalid weights or radii")
        record("validate", "passed", f"T={len(x)}, n={x.shape[1]}, d={x.shape[2]}")
    except Exception as error:
        RUN_OK = False
        record("validate", "failed", repr(error))
else:
    record("validate", "skipped", BOOTSTRAP_ERROR)


## 3. Run EXP-10 in chronological order


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    command = [sys.executable, str(CODE / "kaggle" / "run_application.py"),
               "--data", str(DATA), "--eta", "0.05", "--output-root", str(RESULTS)]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-5000:])
    if outcome.returncode == 0:
        checkpoints = sorted((RESULTS / "application").rglob("metadata.json"))
        if len(checkpoints) != 1:
            RUN_OK = False
            record("application", "failed", f"Expected one checkpoint, found {checkpoints}")
        else:
            ROUND_PATH = checkpoints[0].with_name("rounds.csv")
            record("application", "passed", f"seconds={time.perf_counter()-started:.2f}")
    else:
        RUN_OK = False
        record("application", "failed", f"exit={outcome.returncode}")
else:
    record("application", "skipped", "earlier gate failed")


## 4. Compute conditional population-band widths


In [ ]:
import base64
CERTIFIER_SOURCE = base64.b64decode("".join([
    'IiIiQ29tcHV0ZSBFWFAtMTAgdW5pZm9ybSBwb3B1bGF0aW9uIGJhbmRzIHVuZGVyIGNvbmRpdGlvbmFsIHdpdGhpbi1iYXRj',
    'aCBJSUQuCgpUaGUgVUNJIGNocm9ub2xvZ3kgZG9lcyBub3QgdmVyaWZ5IHRoaXMgYXNzdW1wdGlvbi4gVGhlIHNjcmlwdCBt',
    'ZWFzdXJlcyBiYW5kCndpZHRoLCBub3QgcG9wdWxhdGlvbiBjb3ZlcmFnZSwgYmVjYXVzZSB0aGUgcG9wdWxhdGlvbiB0YXJn',
    'ZXRzIGFyZSB1bmtub3duLgpTZWUgTWF0aC9zdHJlYW1pbmdfcG9wdWxhdGlvbl9pbnRlcnZhbHMubWQgZm9yIHRoZSBjb25k',
    'aXRpb25hbCBwcm9vZi4KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQganNvbgppbXBvcnQgbWF0aApm',
    'cm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKCkZFQVRVUkVTID0gNwpTQ09SRV9CT1VORCA9',
    'IDEyLjAKR0FQX0JPVU5EID0gMTQuMAoKCmRlZiBsb3NzX3JhZGl1cyhuOiBpbnQsIGhvcml6b246IGludCwgZGVsdGFfbG9z',
    'czogZmxvYXQpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdF06CiAgICBpZiBuIDwgMSBvciBob3Jpem9uIDwgMSBvciBub3QgMCA8',
    'IGRlbHRhX2xvc3MgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgY2VydGlmaWNhdGUgcGFyYW1ldGVy',
    'cyIpCiAgICAjIFNlbGVjdGlvbiBpcyBkZXRlcm1pbmlzdGljIGZyb20gcHVibGljIHNhbXBsZSBzaXplIGFuZCBjb25maWRl',
    'bmNlLgogICAgc3BhY2luZ3MgPSBucC5nZW9tc3BhY2UoMC4wMDEsIDEuMCwgMTAwMCkKICAgIGJlc3QgPSAoZmxvYXQoImlu',
    'ZiIpLCBmbG9hdCgibmFuIikpCiAgICBmb3Igc3BhY2luZyBpbiBzcGFjaW5nczoKICAgICAgICBsb2dfbmV0ID0gKEZFQVRV',
    'UkVTICogbnAubG9nKG5wLmNlaWwoMiAvIHNwYWNpbmcpICsgMSkKICAgICAgICAgICAgICAgICAgICsgbnAubG9nKG5wLmNl',
    'aWwoMTAgLyBzcGFjaW5nKSArIDEpKQogICAgICAgIHJhZGl1cyA9IFNDT1JFX0JPVU5EICogbnAuc3FydCgobnAubG9nKDIg',
    'KiBob3Jpem9uIC8gZGVsdGFfbG9zcykgKyBsb2dfbmV0KSAvICgyICogbikpCiAgICAgICAgcmFkaXVzICs9IDggKiBzcGFj',
    'aW5nCiAgICAgICAgaWYgcmFkaXVzIDwgYmVzdFswXToKICAgICAgICAgICAgYmVzdCA9IChmbG9hdChyYWRpdXMpLCBmbG9h',
    'dChzcGFjaW5nKSkKICAgIHJldHVybiBtaW4oU0NPUkVfQk9VTkQsIGJlc3RbMF0pLCBiZXN0WzFdCgoKZGVmIGdhcF9yYWRp',
    'dXMobjA6IGludCwgbjE6IGludCwgaG9yaXpvbjogaW50LCBkZWx0YV9nYXA6IGZsb2F0KSAtPiBmbG9hdDoKICAgIGlmIG1p',
    'bihuMCwgbjEpIDwgMToKICAgICAgICByZXR1cm4gR0FQX0JPVU5ECiAgICBpZiBob3Jpem9uIDwgMSBvciBub3QgMCA8IGRl',
    'bHRhX2dhcCA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW52YWxpZCBjZXJ0aWZpY2F0ZSBwYXJhbWV0ZXJzIikK',
    'ICAgIGxvZ190ZXJtID0gbnAubG9nKDQgKiBGRUFUVVJFUyAqIGhvcml6b24gLyBkZWx0YV9nYXApCiAgICBlcHMwID0gbnAu',
    'c3FydCgyICogbG9nX3Rlcm0gLyBuMCkKICAgIGVwczEgPSBucC5zcXJ0KDIgKiBsb2dfdGVybSAvIG4xKQogICAgcmV0dXJu',
    'IG1pbihHQVBfQk9VTkQsIEZFQVRVUkVTICogKGVwczAgKyBlcHMxKSkKCgpkZWYgY2VydGlmeShkYXRhX3BhdGg6IFBhdGgs',
    'IHJvdW5kc19wYXRoOiBQYXRoLCBvdXRwdXQ6IFBhdGgsIGRlbHRhOiBmbG9hdCA9IDAuMDUpIC0+IGRpY3Q6CiAgICBpZiBu',
    'b3QgMCA8IGRlbHRhIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZWx0YSBtdXN0IGxpZSBpbiAoMCwxKSIpCiAg',
    'ICB3aXRoIG5wLmxvYWQoZGF0YV9wYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGRhdGE6CiAgICAgICAgZ3JvdXBzID0g',
    'bnAuYXNhcnJheShkYXRhWyJncm91cHMiXSwgaW50KQogICAgICAgIHdlaWdodHMgPSBucC5hc2FycmF5KGRhdGFbIndlaWdo',
    'dHMiXSwgZmxvYXQpCiAgICAgICAgc2hhcGUgPSBkYXRhWyJmZWF0dXJlcyJdLnNoYXBlCiAgICBpZiBsZW4oc2hhcGUpICE9',
    'IDMgb3Igc2hhcGVbMl0gIT0gMjQgb3IgZ3JvdXBzLnNoYXBlICE9IHNoYXBlWzoyXSBvciB3ZWlnaHRzLnNoYXBlICE9IChz',
    'aGFwZVswXSwgMyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhpcyBjZXJ0aWZpY2F0ZSByZXF1aXJlcyB0aGUgRVhQ',
    'LTEwIHNpZ25lZC1wYWlyIGZlYXR1cmUgbWFwIikKICAgIHdpdGggcm91bmRzX3BhdGgub3BlbihuZXdsaW5lPSIiLCBlbmNv',
    'ZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgcm91bmRzID0gbGlzdChjc3YuRGljdFJlYWRlcihoYW5kbGUpKQog',
    'ICAgaG9yaXpvbiwgbiwgXyA9IHNoYXBlCiAgICBpZiBsZW4ocm91bmRzKSAhPSBob3Jpem9uOgogICAgICAgIHJhaXNlIFZh',
    'bHVlRXJyb3IoInJvdW5kIGNvdW50IG1pc21hdGNoIikKICAgIHJpc2tfcmFkaXVzLCBzcGFjaW5nID0gbG9zc19yYWRpdXMo',
    'biwgaG9yaXpvbiwgZGVsdGEgLyAyKQogICAgcm93cyA9IFtdCiAgICBmb3IgdCBpbiByYW5nZShob3Jpem9uKToKICAgICAg',
    'ICBuMCA9IGludChucC5zdW0oZ3JvdXBzW3RdID09IDApKQogICAgICAgIG4xID0gaW50KG5wLnN1bShncm91cHNbdF0gPT0g',
    'MSkpCiAgICAgICAgZ2FwID0gZ2FwX3JhZGl1cyhuMCwgbjEsIGhvcml6b24sIGRlbHRhIC8gMikKICAgICAgICBjZW50ZXIg',
    'PSBucC5hc2FycmF5KGpzb24ubG9hZHMocm91bmRzW3RdWyJtaWRwb2ludHMiXSksIGZsb2F0KQogICAgICAgIHJhZGl1cyA9',
    'IG5wLmFycmF5KFtyaXNrX3JhZGl1cywgZ2FwLCAwLjBdKQogICAgICAgIGlmIGNlbnRlci5zaGFwZSAhPSAoMywpIG9yIG5v',
    'dCBucC5hbGwobnAuaXNmaW5pdGUoY2VudGVyKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJpbnZhbGlkIG9i',
    'amVjdGl2ZSBjZW50ZXIgYXQgYmF0Y2gge3QgKyAxfSIpCiAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAiYmF0',
    'Y2giOiB0ICsgMSwgIm4wIjogbjAsICJuMSI6IG4xLAogICAgICAgICAgICAib2JzZXJ2ZWRfbG9nbG9zc19jZW50ZXIiOiBm',
    'bG9hdChjZW50ZXJbMF0pLAogICAgICAgICAgICAib2JzZXJ2ZWRfZ2FwX2NlbnRlciI6IGZsb2F0KGNlbnRlclsxXSksCiAg',
    'ICAgICAgICAgICJyaXNrX3JhZGl1cyI6IGZsb2F0KHJhZGl1c1swXSksICJnYXBfcmFkaXVzIjogZmxvYXQocmFkaXVzWzFd',
    'KSwKICAgICAgICAgICAgInJlc291cmNlX3JhZGl1cyI6IDAuMCwKICAgICAgICAgICAgIndlaWdodGVkX2hhbGZfd2lkdGgi',
    'OiBmbG9hdCh3ZWlnaHRzW3RdIEAgcmFkaXVzKSwKICAgICAgICB9KQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwg',
    'ZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggKG91dHB1dCAvICJwb3B1bGF0aW9uX2JhbmRfd2lkdGhzLmNzdiIpLm9wZW4oInci',
    'LCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0',
    'ZXIoaGFuZGxlLCBmaWVsZG5hbWVzPWxpc3Qocm93c1swXSkpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAg',
    'ICB3cml0ZXIud3JpdGVyb3dzKHJvd3MpCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJzdGF0dXMiOiAiY29uZGl0aW9uYWxf',
    'Y2VydGlmaWNhdGVfb25seSIsCiAgICAgICAgInNhbXBsaW5nX2Fzc3VtcHRpb24iOiAiY3VycmVudCBiYXRjaCBJSUQgY29u',
    'ZGl0aW9uYWwgb24gcGFzdDsgdW52ZXJpZmllZCBmb3IgVUNJIEJhbmsgTWFya2V0aW5nIiwKICAgICAgICAidGFyZ2V0Ijog',
    'ImNvbmRpdGlvbmFsIHBvcHVsYXRpb24gb2JqZWN0aXZlIGZ1bmN0aW9ucywgdW5pZm9ybWx5IG92ZXIgYWxsIGJveCBhY3Rp',
    'b25zIiwKICAgICAgICAic2ltdWx0YW5lb3VzX2NvbmZpZGVuY2VfaWZfYXNzdW1wdGlvbl9ob2xkcyI6IDEgLSBkZWx0YSwK',
    'ICAgICAgICAiaG9yaXpvbiI6IGhvcml6b24sICJiYXRjaF9zaXplIjogbiwKICAgICAgICAicmlza19ncmlkX3NwYWNpbmci',
    'OiBzcGFjaW5nLAogICAgICAgICJyaXNrX3JhZGl1cyI6IHJpc2tfcmFkaXVzLAogICAgICAgICJtZWFuX2dhcF9yYWRpdXMi',
    'OiBmbG9hdChucC5tZWFuKFtyb3dbImdhcF9yYWRpdXMiXSBmb3Igcm93IGluIHJvd3NdKSksCiAgICAgICAgIm1heF9nYXBf',
    'cmFkaXVzIjogZmxvYXQobnAubWF4KFtyb3dbImdhcF9yYWRpdXMiXSBmb3Igcm93IGluIHJvd3NdKSksCiAgICAgICAgIm1l',
    'YW5fb2JzZXJ2ZWRfbG9nbG9zc19jZW50ZXIiOiBmbG9hdChucC5tZWFuKFtyb3dbIm9ic2VydmVkX2xvZ2xvc3NfY2VudGVy',
    'Il0gZm9yIHJvdyBpbiByb3dzXSkpLAogICAgICAgICJtZWFuX29ic2VydmVkX2dhcF9jZW50ZXIiOiBmbG9hdChucC5tZWFu',
    'KFtyb3dbIm9ic2VydmVkX2dhcF9jZW50ZXIiXSBmb3Igcm93IGluIHJvd3NdKSksCiAgICAgICAgIm1lYW5fd2VpZ2h0ZWRf',
    'aGFsZl93aWR0aCI6IGZsb2F0KG5wLm1lYW4oW3Jvd1sid2VpZ2h0ZWRfaGFsZl93aWR0aCJdIGZvciByb3cgaW4gcm93c10p',
    'KSwKICAgICAgICAiVEhNMDJfVV9UX3VwcGVyX2JvdW5kIjogZmxvYXQoMiAqIG5wLnN1bShbcm93WyJ3ZWlnaHRlZF9oYWxm',
    'X3dpZHRoIl0gZm9yIHJvdyBpbiByb3dzXSkpLAogICAgICAgICJlbXBpcmljYWxfcG9wdWxhdGlvbl9jb3ZlcmFnZV9tZWFz',
    'dXJhYmxlIjogRmFsc2UsCiAgICB9CiAgICBpZiBucC5hbGxjbG9zZSh3ZWlnaHRzLCBbMC44LCAwLjE1LCAwLjA1XSk6CiAg',
    'ICAgICAgZ3JhZGllbnRfYm91bmQgPSAwLjg1ICogbWF0aC5zcXJ0KDI0KSArIDAuMTUgKiAyICogbWF0aC5zcXJ0KDE0KQog',
    'ICAgICAgIG9nZF9zdGF0aWNfcGFydCA9IDI0IC8gKDIgKiAwLjA1KSArIDAuMDUgKiBncmFkaWVudF9ib3VuZCoqMiAqIGhv',
    'cml6b24gLyAyCiAgICAgICAgc3VtbWFyeVsic3RhdGljX2NvbXBhcmF0b3JfZW52ZWxvcGVfYXRfZXRhXzAuMDUiXSA9IHsK',
    'ICAgICAgICAgICAgImdyYWRpZW50X25vcm1fYm91bmQiOiBncmFkaWVudF9ib3VuZCwKICAgICAgICAgICAgIm9wdGltaXph',
    'dGlvbl9wYXJ0Ijogb2dkX3N0YXRpY19wYXJ0LAogICAgICAgICAgICAidG90YWxfd2l0aF9VX1QiOiBvZ2Rfc3RhdGljX3Bh',
    'cnQgKyBzdW1tYXJ5WyJUSE0wMl9VX1RfdXBwZXJfYm91bmQiXSwKICAgICAgICAgICAgImludGVycHJldGF0aW9uIjogImNv',
    'bmRpdGlvbmFsIHdvcnN0LWNhc2UgYm91bmQsIG5vdCBtZWFzdXJlZCByZWdyZXQiLAogICAgICAgIH0KICAgIChvdXRwdXQg',
    'LyAicG9wdWxhdGlvbl9iYW5kX3N1bW1hcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9',
    'MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gc3VtbWFyeQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAg',
    'ICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCB0',
    'eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJvdW5kcyIsIHR5cGU9UGF0aCwg',
    'cmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1U',
    'cnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZWx0YSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wNSkKICAgIGFy',
    'Z3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBwcmludChqc29uLmR1bXBzKGNlcnRpZnkoYXJncy5kYXRhLCBhcmdzLnJv',
    'dW5kcywgYXJncy5vdXRwdXQsIGFyZ3MuZGVsdGEpLCBpbmRlbnQ9MikpCg==',
]))
PROOF_TEXT = base64.b64decode("".join([
    'IyBDb25kaXRpb25hbCBwb3B1bGF0aW9uIGludGVydmFscyBmb3IgRVhQLTEwCgoqKlN0YXR1czogY29uZGl0aW9uYWwgcHJv',
    'cG9zaXRpb24sIHByb3ZlZC4qKiBUaGlzIGlzIGEgc2VwYXJhdGUgc3RvY2hhc3RpYwppbnRlcnByZXRhdGlvbiBvZiBFWFAt',
    'MTAuIFRoZSBwdWJsaXNoZWQgZGF0ZSBvcmRlciBvZiBVQ0kgQmFuayBNYXJrZXRpbmcgZG9lcwpub3QgdmVyaWZ5IGl0cyBz',
    'YW1wbGluZyBhc3N1bXB0aW9uLgoKIyMgVGFyZ2V0IGFuZCBhc3N1bXB0aW9ucwoKQXQgcm91bmQgYHRgLCBsZXQgYEhfe3Qt',
    'MX1gIGNvbnRhaW4gZXZlcnkgZWFybGllciBiYXRjaCBhbmQgdGhlIHBhc3Qtb25seQpmZWF0dXJlIHRyYW5zZm9ybS4gQ29u',
    'ZGl0aW9uYWwgb24gYEhfe3QtMX1gLCB0aGUgMjU2IG9ic2VydmF0aW9ucwpgKFNfaSxZX2ksR19pKWAgaW4gdGhlIGN1cnJl',
    'bnQgYmF0Y2ggYXJlIGluZGVwZW5kZW50IGFuZCBpZGVudGljYWxseQpkaXN0cmlidXRlZCBmcm9tIGEgZGlzdHJpYnV0aW9u',
    'IGBQX3RgLiBUaGUgZGlzdHJpYnV0aW9uIG1heSBjaGFuZ2Ugd2l0aCBgdGAKYW5kIG1heSBkZXBlbmQgb24gYEhfe3QtMX1g',
    'LiBBc3N1bWUgYFPiiIhbLTEsMV1eN2AsIGBZ4oiIey0xLCsxfWAgYW5kIGJvdGggYWdlCmdyb3VwcyBoYXZlIHBvc2l0aXZl',
    'IHByb2JhYmlsaXR5LiBUaGVzZSBhc3N1bXB0aW9ucyBjb25jZXJuIHRoZSB0cmFuc2Zvcm1lZApzZXZlbiBwcmUtY2FsbCB2',
    'YXJpYWJsZXMsIG5vdCB0aGUgc2lnbmVkLXBhaXIgcmVwcmVzZW50YXRpb24uCgpGb3IgYSBib3ggYWN0aW9uIGB34oiIWzAs',
    'MV1eMjRgLCB0aGUgZXhpc3RpbmcgZmVhdHVyZSBtYXAgaW5kdWNlcyBlZmZlY3RpdmUKY29lZmZpY2llbnRzIGDOuCh3KeKI',
    'iFstMSwxXV43YCBhbmQgaW50ZXJjZXB0IGBiKHcp4oiIWy01LDVdYCwgd2l0aCBzY29yZQpgel93KFMpPc64KHcpwrdTK2Io',
    'dylgLiBIZW5jZSBgfHpfdyhTKXziiaQxMmAuIERlZmluZSBwb3B1bGF0aW9uIG9iamVjdGl2ZXMKCmBoX3QsMCh3KT1FX3tQ',
    'X3R9W2xvZygxK2V4cCgtWSB6X3coUykpKV1gLAoKYGhfdCwxKHcpPXxFX3tQX3R9W3pfdyhTKXxHPTBdLUVfe1BfdH1bel93',
    'KFMpfEc9MV18YCwgYW5kCgpgaF90LDIodyk9fHx3fHzCsi8yYC4KCkxldCBgY190LGpgIGJlIHRoZSBjb3JyZXNwb25kaW5n',
    'IGN1cnJlbnQtYmF0Y2ggb2JqZWN0aXZlcy4gQm90aCBgaF90LGpgIGFuZApgY190LGpgIGFyZSBjb252ZXggaW4gYHdgLiBU',
    'aGUgaW50ZXJjZXB0IGNhbmNlbHMgZnJvbSB0aGUgZ3JvdXAgZ2FwLiBUaGUKY3VycmVudCBiYXRjaCwgaXRzIG9iamVjdGl2',
    'ZXMsIGFuZCBpbnRlcnZhbCBlbmRwb2ludHMgYmVjb21lIGF2YWlsYWJsZSBvbmx5CmFmdGVyIHRoZSBsZWFybmVyIGhhcyBz',
    'ZWxlY3RlZCBgd190YDsgdGhpcyBtYXRjaGVzIHRoZSBwb3N0LWRlY2lzaW9uIGZlZWRiYWNrCnByb3RvY29sLiBUaGUgbGVh',
    'cm5lcidzIGFjdGlvbiBjYW4gZGVwZW5kIG9uIHRoZSBwYXN0IGJ1dCBub3QgdGhlIGN1cnJlbnQKYmF0Y2guCgojIyBTaW11',
    'bHRhbmVvdXMgaW50ZXJ2YWwgY29uc3RydWN0aW9uCgpGaXggYM604oiIKDAsMSlgLCBob3Jpem9uIGBUYCwgYW5kIHNwbGl0',
    'IGDOtD3OtF8wK860XzFgLiBTZWxlY3QgYSBkZXRlcm1pbmlzdGljCmdyaWQgc3BhY2luZyBgYeKIiCgwLDFdYCB1c2luZyBv',
    'bmx5IHB1YmxpYyBgbixULM60YC4gQSBjb29yZGluYXRlIGdyaWQgb2YKYFstMSwxXV43w5dbLTUsNV1gIHdpdGggbWF4aW11',
    'bSBzcGFjaW5nIGBhYCBjb250YWlucyBhdCBtb3N0CgpgTihhKT0oY2VpbCgyL2EpKzEpXjcgKGNlaWwoMTAvYSkrMSlgIHBv',
    'aW50cy4KCkxldCBgbl90YCBiZSB0aGUgYmF0Y2ggc2l6ZSBhbmQgYG5fdGdgIGl0cyBncm91cCBjb3VudHMuIERlZmluZQoK',
    'YHJfdCwwKGEpPW1pbnsxMiwgMTIgc3FydChsb2coMiBOKGEpIFQvzrRfMCkvKDJuX3QpKSs4YX1gLAoKYGVfdGc9c3FydCgy',
    'IGxvZygyOFQvzrRfMSkvbl90ZylgIGZvciBgbl90Zz4wYCwKCmByX3QsMT1taW57MTQsIDcoZV90MCtlX3QxKX1gLCBhbmQg',
    'YHJfdCwyPTBgLgoKSWYgYSBncm91cCBpcyBhYnNlbnQsIHNldCBgY190LDE9MGAgYW5kIGByX3QsMT0xNGAgZm9yIHRoYXQg',
    'cm91bmQuIFNldApgTF90LGoodyk9Y190LGoodyktcl90LGpgIGFuZCBgVV90LGoodyk9Y190LGoodykrcl90LGpgLiBVbmRl',
    'ciB0aGUgYXNzdW1wdGlvbnMKYWJvdmUsIHdpdGggcHJvYmFiaWxpdHkgYXQgbGVhc3QgYDEtzrRgLCAqKnNpbXVsdGFuZW91',
    'c2x5IGZvciBldmVyeSByb3VuZCwKb2JqZWN0aXZlLCBhbmQgYWN0aW9uKiosIGBMX3Qsaih3KeKJpGhfdCxqKHcp4omkVV90',
    'LGoodylgLiBUaGUgY29uc3RhbnQgcmFkaWkKcHJlc2VydmUgY29udmV4aXR5IG9mIGJvdGggZW5kcG9pbnRzIGFuZCBsZWF2',
    'ZSB0aGUgbWlkcG9pbnQgZ3JhZGllbnRzCnVuY2hhbmdlZC4gVGhlIGtub3duIHJlc291cmNlIG9iamVjdGl2ZSBoYXMgZXhh',
    'Y3QgemVybyB3aWR0aC4KCiMjIyBQcm9vZgoKRm9yIGV2ZXJ5IGZpeGVkIGdyaWQgYWN0aW9uLCBsb2dpc3RpYyBsb3NzIGhh',
    'cyBhIHJhbmdlIG9mIGxlbmd0aCAxMjogaXRzCm1hcmdpbiBsaWVzIGluIGBbLTEyLDEyXWAsIGFuZApgbG9nKDErZXhwKDEy',
    'KSktbG9nKDErZXhwKC0xMikpPTEyYC4gQ29uZGl0aW9uYWwgSG9lZmZkaW5nIGdpdmVzCmBQcih8Y190LDAtaF90LDB8PnEg',
    'fCBIX3t0LTF9KeKJpDIgZXhwKC0ybl90IHHCsi8xMsKyKWAgYXQgdGhhdCBhY3Rpb24uCkFwcGx5IGEgdW5pb24gYm91bmQg',
    'dG8gdGhlIGBOKGEpYCBncmlkIHBvaW50cyBhbmQgYFRgIHJvdW5kcywgYWxsb2NhdGluZwpgzrRfMGAuIExvZ2lzdGljIGxv',
    'c3MgaXMgMS1MaXBzY2hpdHogaW4gc2NvcmU7IHRoZSBlbXBpcmljYWwgYW5kIHBvcHVsYXRpb24Kcmlza3MgYXJlIGVhY2gg',
    'MS1MaXBzY2hpdHogaW4gYGxfMWAgZGlzdGFuY2Ugb2YgYCjOuCxiKWAuIEEgbmVhcmVzdCBncmlkIHBvaW50CmlzIHdpdGhp',
    'biBgNGFgIGluIHRoYXQgZGlzdGFuY2UsIHNvIHRoZSBkaXNjcmVwYW5jeSBhdCBhbiBhcmJpdHJhcnkgYWN0aW9uCmlzIHdp',
    'dGhpbiBgOGFgIG9mIHRoZSBncmlkIGRpc2NyZXBhbmN5LiBTaW5jZSB0d28gZXhwZWN0YXRpb25zIG9mIHZhcmlhYmxlcwpp',
    'biB0aGUgc2FtZSBsZW5ndGgtMTIgcmFuZ2UgY2FuIGRpZmZlciBieSBhdCBtb3N0IDEyLCB0aGUgbWluaW11bSB3aXRoIDEy',
    'CmFsc28gcHJlc2VydmVzIHRoZSBib3VuZC4KCkNvbmRpdGlvbmFsIG9uIGEgYmF0Y2gncyBwb3NpdGl2ZSBncm91cCBjb3Vu',
    'dHMsIHRoZSBvYnNlcnZhdGlvbnMgaW4gZWFjaApncm91cCBhcmUgaWlkIGZyb20gYFBfdChTfEc9ZylgLiBFYWNoIGNvb3Jk',
    'aW5hdGUgbGllcyBpbiBgWy0xLDFdYC4KSG9lZmZkaW5nIGFuZCBhIHVuaW9uIGJvdW5kIG92ZXIgdHdvIGdyb3Vwcywgc2V2',
    'ZW4gY29vcmRpbmF0ZXMsIGFuZCBgVGAKcm91bmRzIHNob3cgdGhhdCBhbGwgZ3JvdXAtY29vcmRpbmF0ZSBtZWFuIGVycm9y',
    'cyBhcmUgYXQgbW9zdCBgZV90Z2Agd2l0aApwcm9iYWJpbGl0eSBhdCBsZWFzdCBgMS3OtF8xYC4gRm9yIGV2ZXJ5IGB8zrhf',
    'anziiaQxYCwgdGhlIGRpZmZlcmVuY2Ugb2YgdGhlCnBvcHVsYXRpb24gYW5kIGVtcGlyaWNhbCBzaWduZWQgZ3JvdXAgY29u',
    'dHJhc3RzIGlzIGF0IG1vc3QKYDcoZV90MCtlX3QxKWAuIFRoZSBhYnNvbHV0ZS12YWx1ZSBtYXAgaXMgMS1MaXBzY2hpdHos',
    'IHNvIHRoaXMgYWxzbyBib3VuZHMKdGhlIGdhcC1vYmplY3RpdmUgZGlzY3JlcGFuY3kuIEJvdGggZ2FwcyBsaWUgaW4gYFsw',
    'LDE0XWAsIGFsbG93aW5nIHRoZSBjYXAKYXQgMTQuIFRoZSBlbXB0eS1ncm91cCBmYWxsYmFjayBjb3ZlcnMgdGhhdCBjYXNl',
    'IGRldGVybWluaXN0aWNhbGx5LiBUaGUKcmVzb3VyY2Ugb2JqZWN0aXZlIGlzIGlkZW50aWNhbCBpbiBzYW1wbGUgYW5kIHBv',
    'cHVsYXRpb24uIEZpbmFsbHksIHVuaW9uCnRoZSByaXNrIGFuZCBncm91cCBldmVudHMuIFRoZSBhcmd1bWVudCBpcyBjb25k',
    'aXRpb25hbCBhdCBlYWNoIHJvdW5kIGFuZAp0aGVuIHVzZXMgdGhlIHRvd2VyIHByb3BlcnR5OyBpdCBkb2VzIG5vdCByZXF1',
    'aXJlIHN0YXRpb25hcnkgYFBfdGAuIOKWoQoKIyMgSW50ZXJwcmV0YXRpb24KClRoaXMgY29uc3RydWN0aW9uIHN1cHBsaWVz',
    'IEE0IHdpdGggY29uZmlkZW5jZSBgMS3OtGAgdW5kZXIgaXRzIHNhbXBsaW5nCmFzc3VtcHRpb24sIHVuaWZvcm1seSBvdmVy',
    'IGhpbmRzaWdodCBjb21wYXJhdG9yIGFjdGlvbnMuIEl0IGFsc28gc3VwcGxpZXMKY29udmV4IG9yZGVyZWQgZW5kcG9pbnRz',
    'IGZvciBBMyBhbmQgcG9zdC1kZWNpc2lvbiBtaWRwb2ludCBncmFkaWVudHMgZm9yIEE1LgpUSE0tMDIgdGhlbiBhcHBsaWVz',
    'IG9uIHRoZSBzaW11bHRhbmVvdXMtY29udGFpbm1lbnQgZXZlbnQsIHN1YmplY3QgdG8gaXRzCnJlbWFpbmluZyBhc3N1bXB0',
    'aW9ucy4gVGhpcyBpcyBhICoqaGlnaC1wcm9iYWJpbGl0eSBjb25kaXRpb25hbCB0aGVvcmVtKiosIG5vdAphbiB1bmNvbmRp',
    'dGlvbmFsIGNsYWltIGFib3V0IHRoZSBVQ0kgc3RyZWFtLiBJZiBvYnNlcnZhdGlvbnMgd2l0aGluIGEgYmF0Y2gKYXJlIGRl',
    'cGVuZGVudCwgb3IgaWYgdGhlIHJlcG9ydGVkIHJvdyBvcmRlciBkb2VzIG5vdCBjb3JyZXNwb25kIHRvIGRyYXdzCmZyb20g',
    'YSBzaW5nbGUgYFBfdGAsIHRoaXMgcHJvb2YgZ2l2ZXMgbm8gY292ZXJhZ2UgZ3VhcmFudGVlLiBXaWRlIGludGVydmFscwpt',
    'YXkgbWFrZSB0aGUgcmVzdWx0aW5nIHJlZ3JldCBlbnZlbG9wZSB1bmluZm9ybWF0aXZlLgoKRm9yIHRoZSBFWFAtMTAgd2Vp',
    'Z2h0cyBgKDAuOCwwLjE1LDAuMDUpYCwgYSB2YWxpZCBtaWRwb2ludCBzdWJncmFkaWVudCBub3JtCmJvdW5kIGlzIGBHPTAu',
    'ODUgc3FydCgyNCkrMC4zMCBzcXJ0KDE0KWA6IGxvZ2lzdGljIGFuZCByZXNvdXJjZSBncmFkaWVudHMKZWFjaCBoYXZlIG5v',
    'cm0gYXQgbW9zdCBgc3FydCgyNClgOyB0aGUgYWdlLWdhcCBzdWJncmFkaWVudCBoYXMgbm9ybSBhdCBtb3N0CmAyIHNxcnQo',
    'MTQpYCBiZWNhdXNlIHRoZSB0ZW4gYmlhcyBjb29yZGluYXRlcyBjYW5jZWwuIFRoZSBib3ggZGlhbWV0ZXIgaXMKYEQ9c3Fy',
    'dCgyNClgLiBUaGVzZSBjb25zdGFudHMgcGVybWl0IGEgZnVsbHkgbnVtZXJpYyBUSE0tMDIgZW52ZWxvcGUgb25jZQp0aGUg',
    'Y29uZGl0aW9uYWwgcmFkaWkgYXJlIGNvbXB1dGVkLiBUaGV5IGRvIG5vdCBtYWtlIGl0IHRpZ2h0Lgo=',
]))
(CODE / "experiments" / "certify_bank_population.py").write_bytes(CERTIFIER_SOURCE)
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "streaming_population_intervals.md").write_bytes(PROOF_TEXT)
if RUN_OK:
    command = [sys.executable, str(CODE / "experiments" / "certify_bank_population.py"),
               "--data", str(DATA), "--rounds", str(ROUND_PATH),
               "--output", str(RESULTS / "population_certificate"), "--delta", "0.05"]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-6000:])
    if outcome.returncode == 0:
        CERTIFICATE = json.loads((RESULTS / "population_certificate" /
                                  "population_band_summary.json").read_text(encoding="utf-8"))
        record("population_certificate", "passed", "conditional widths saved; IID assumption unverified")
    else:
        RUN_OK = False
        record("population_certificate", "failed", f"exit={outcome.returncode}")
else:
    record("population_certificate", "skipped", "earlier gate failed")


## 5. Zip the proof and outputs


In [ ]:
import zipfile
(RESULTS / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
if BOOTSTRAP_ERROR is None:
    shutil.copy2(SOURCE_META, RESULTS / "bank_stream.metadata.json")
FINAL_ZIP = WORK_ROOT / "result_jcam_exp10_population_bands.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=(Path("exp10_population_bands") / path.relative_to(RESULTS)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
